In [32]:
import os
import glob
import io

folder = '/Users/lucyford/Desktop/UOM CCL/Semester 2/Corpus Linguistics/BNC/Texts'

try:
    from google.colab import files
    uploaded_files = files.upload()
except ImportError:
    uploaded_files = {}
    for file_path in glob.glob(os.path.join(folder, '*')):
        if os.path.isfile(file_path):
            filename = os.path.basename(file_path)
            uploaded_files[filename] = open(file_path, 'rb').read()
print(f"已加载 {len(uploaded_files)} 个文件：")

Saving CPT.xml to CPT.xml
已加载 1 个文件：


In [33]:
import xml.etree.ElementTree as ET
import pandas as pd

In [35]:
# Create KWIC concordance lines for "help" from our XML files

help_KWIC = []
help_form = []
help_pos = []

max_buffer = 30  # for left context, keep last 30 words

# Metadata lists
file_id = []
genre = []
mode = []
subgenre = []
year = []
hits = []
hit = 1
corpus = []
variety = []

# Speaker data (for Spoken texts only, "NA" for Written)
speaker_id = []
speaker_gender = []
speaker_age = []
soc = []

dep_var = []

# BNC age group codes → midpoint of age range
# Actual codes found in BNC: Ag0 (unknown), Ag1-Ag5, X (unknown)
age_midpoint = {
    'Ag0': 'unknown',
    'Ag1': 7,    # 0-14
    'Ag2': 20,   # 15-24
    'Ag3': 30,   # 25-34
    'Ag4': 40,   # 35-44
    'Ag5': 55,   # 45+
    'X':   'unknown',
}

# XML namespace for xml:id attribute
XML_NS = '{http://www.w3.org/XML/1998/namespace}'

# Helper function: determine DepVar from elements after "help"
# Uses BNC c5 tags: TO0=to-infinitive marker, VVI=bare infinitive, VVG=-ing form
def get_dep_var(elements, help_idx, help_c5):
    if not help_c5 or not help_c5.startswith(('VV', 'NN1-VVG')): #added last condition for gerund -ing forms
        return 'NA'
    STOP_C5   = {'VVD', 'VVZ', 'VHD', 'VHZ', 'VBD', 'VBZ', 'VM0'}
    STOP_POS  = {'PNQ', 'CJS', 'CJT'}
    STOP_TEXT = {'and', 'or', 'if', 'whether', 'which', 'who', 'whom', 'that', 'with', 'by'}
    word_count = 0
    i = help_idx + 1
    while i < len(elements) and word_count < 15:
        elem = elements[i]
        elem_plusone = elements[i+1] if (i+1)<len(elements) else None
        i += 1
        if elem.tag not in ['w', 'c']:
            continue
        c5   = elem.get('c5', '')
        c5_plusone=elem_plusone.get('c5', '') if elem_plusone is not None else None
        text = (elem.text or '').strip().lower()
        if elem.tag == 'w':
            word_count += 1
        # Stop at sentence-ending punctuation
        if text in {'.', '!', '?', ','}:
            break
        # Stop if punctuation is attached to the word token (e.g. "them,")
        if elem.tag == 'w' and text.endswith((',', '.', '!', '?')):
            break
        # Stop at finite verbs, modals, or subordinating conjunctions
        if elem.tag == 'w' and (c5 in STOP_C5 or c5 in STOP_POS or text in STOP_TEXT):
            break
        # TO infinitive: "to (adv/neg)* VVI/VVB"
        if elem.tag == 'w' and (c5 == 'TO0' or text == 'to') or (c5 in {'PNP', 'PNX'} and c5_plusone == 'TO0'):
            for j in range(i, min(i + 6, len(elements))):
                nxt = elements[j]
                if nxt.tag != 'w':
                    continue
                nxt_c5 = nxt.get('c5', '')
                if nxt_c5 in {'AV0', 'XX0', 'TO0'}: #add TO0 for spoken examples with repeated examples
                    continue
                if nxt_c5 in {'VVI', 'VVB', 'VDI', 'VDB', 'VBI', 'VHI', 'VHB', 'NN1-VVB','TO0'}:
                    return 'TO'
            break
        # INING: "in (det/pron/noun)* VVG"
        if elem.tag == 'w' and text == 'in':
            for j in range(i, min(i + 4, len(elements))):
                nxt = elements[j]
                if nxt.tag != 'w':
                    continue
                if nxt.get('c5', '') in {'VVG', 'VBG', 'VDG', 'VHG'}:
                    return 'INING'
                if nxt.get('c5', '') in {'AT0', 'DT0', 'PNP', 'NN0', 'NN1', 'NN2'}:
                    continue
                break
        # ING: -ing form directly after help
        #add tags for be, do, have?
        if elem.tag == 'w' and c5 == 'VVG': # add tag for 'doing'
          #initialise flag for excluding 'by' examples e.g. "helps by doing"
          found_by=False
          for j in range(i, min(i + 6, len(elements))):
                nxt = elements[j]
                if nxt.tag != 'w': #this should control for sentence breaks with <s> too
                  continue
                if nxt.text == 'by'.lower():
                  found_by=True
                  break
          if not found_by:
            continue
          return 'ING'
          #personal pronouns:
        if elem.tag =='w' and c5 in {'PNP', 'PNX'} and c5_plusone.startswith('VVG'): #use startswith to also capture ambiguity tags
          return 'ING'
        # BARE: bare infinitive directly after help
        if elem.tag == 'w' and c5 in {'VVI', 'VVB', 'VDI', 'VDB', 'VBI', 'VHB', 'VHI', 'NN1-VVB'}:
          return 'BARE'
          #personal pronounds:
        if elem.tag =='w' and c5 in {'PNP', 'PNX'} and c5_plusone.startswith(('VVI', 'VVB', 'VDI', 'VDB', 'VBI', 'VHB', 'VHI', 'NN1-VVB')):
          return 'BARE'
    return 'NA'


for file, content in uploaded_files.items():
    tree = ET.parse(io.BytesIO(content))

    # --- (1) File-level metadata ---

    id = file.replace('.xml', '')

    classcode = tree.find('.//classCode')
    meta = classcode.text

    creation = tree.find('.//creation')
    yearvalue = creation.get('date')
    if yearvalue == '0000':
        yearvalue = '1990'

    # Determine mode and extract mode-specific metadata once per file
    if meta[0] == "W":
        modevalue = "Written"
        wstext_element = tree.find('.//wtext') or tree.find('.//stext')
        genrevalue = wstext_element.get('type') if wstext_element is not None else "NA"
        subgenrevalue = meta[2:]
        spk_id, spk_gender, spk_age, spk_soc = "NA", "NA", "NA", "NA"

    elif meta[0] == "S":
        modevalue = "Spoken"
        genrevalue = "NA"
        subgenrevalue = "NA"
        person = tree.find('.//person')
        if person is not None:
            spk_id = person.get(f'{XML_NS}id', 'NA')   # fix: use XML namespace
            spk_gender = person.get('sex', 'NA')
            raw_age = person.get('ageGroup', 'Ag0')
            spk_age = age_midpoint.get(raw_age, 'unknown')  # fix: Ag1-Ag5 codes
            spk_soc = person.get('soc', 'NA')
        else:
            spk_id, spk_gender, spk_age, spk_soc = "NA", "NA", "unknown", "NA"

    else:
        modevalue = "Unknown"
        wstext_element = tree.find('.//wtext') or tree.find('.//stext')
        genrevalue = wstext_element.get('type') if wstext_element is not None else "NA"
        subgenrevalue = meta[2:]
        spk_id, spk_gender, spk_age, spk_soc = "NA", "NA", "NA", "NA"

    # --- (2) KWIC concordance ---

    left_context = []
    collecting_right = False
    right_context = []
    current_right_context_words = 0

    all_elems = list(tree.iter())

    for elem_idx, elem in enumerate(all_elems):

        # If we're collecting right context (after a "help" hit)
        if collecting_right:
            if elem.tag in ['w', 'c']:
                right_context.append(elem.text)
                if elem.tag == 'w':
                    current_right_context_words += 1
                    if current_right_context_words >= 60:
                        collecting_right = False
                        right_KWIC = ' '.join(filter(None, right_context))
                        help_KWIC.append(f"{left_KWIC} {help_word} {right_KWIC}")

        # Build left context buffer
        if elem.tag in ['w', 'c'] and elem.text and not elem.text.lower().startswith('help'):
            left_context.append(elem.text)
            if len(left_context) > max_buffer:
                left_context.pop(0)

        # Found a "help" hit
        if elem.tag == 'w' and elem.text and elem.text.lower().startswith('help'):

            # If still collecting right context from a previous hit, store it first
            if collecting_right:
                right_KWIC = ' '.join(filter(None, right_context))
                help_KWIC.append(f"{left_KWIC} {help_word} {right_KWIC}")

            left_KWIC = ' '.join(left_context)
            help_word = elem.text
            help_form.append(help_word)
            help_pos.append(elem.get('c5'))

            dep_var.append(get_dep_var(all_elems, elem_idx, elem.get('c5')))

            # Append all metadata for this hit
            file_id.append(id)
            genre.append(genrevalue)
            mode.append(modevalue)
            subgenre.append(subgenrevalue)
            year.append(yearvalue)
            corpus.append("BNC")
            variety.append("BrE")
            speaker_id.append(spk_id)
            speaker_gender.append(spk_gender)
            speaker_age.append(spk_age)
            soc.append(spk_soc)
            hits.append(str(hit))
            hit += 1

            # Start collecting right context
            collecting_right = True
            right_context = []
            current_right_context_words = 0

    # If the last hit's right context wasn't stored yet
    if collecting_right:
        right_KWIC = ' '.join(filter(None, right_context))
        help_KWIC.append(f"{left_KWIC} {help_word} {right_KWIC}")

# --- (3) Build DataFrame and export two Excel files ---

df_all = pd.DataFrame({
    'Hit': hits,
    'KWIC': help_KWIC,
    'Form': help_form,
    'POS': help_pos,
    'DepVar': dep_var,
    'TextID': file_id,
    'Year': year,
    'Genre': genre,
    'Subgenre': subgenre,
    'Mode': mode,
    'Variety': variety,
    'Corpus': corpus,
    'SpeakerID': speaker_id,
    'SpeakerGender': speaker_gender,
    'SpeakerAge': speaker_age,
    'SpeakerSoc': soc
})

# Written: TextID | Year | Genre | Subgenre | Mode | Variety | Corpus
df_written = df_all[df_all['Mode'] == 'Written'][[
    'Hit', 'KWIC', 'Form', 'POS', 'DepVar',
    'TextID', 'Year', 'Genre', 'Subgenre', 'Mode', 'Variety', 'Corpus'
]].reset_index(drop=True)

# Spoken: TextID | SpeakerID | Year | SpeakerGender | SpeakerAge | SpeakerSoc | Mode | Variety | Corpus
df_spoken = df_all[df_all['Mode'] == 'Spoken'][[
    'Hit', 'KWIC', 'Form', 'POS', 'DepVar',
    'TextID', 'SpeakerID', 'Year', 'SpeakerGender', 'SpeakerAge', 'SpeakerSoc', 'Mode', 'Variety', 'Corpus'
]].reset_index(drop=True)

df_written.to_excel(f'BNC_help_written.xlsx', index=False)
df_spoken.to_excel(f'BNC_help_spoken.xlsx', index=False)

print(f"Written: {len(df_written)} results → BNC_help_written.xlsx")
print(f"Spoken:  {len(df_spoken)} results → BNC_help_spoken.xlsx")

Written: 9 results → testCPT.xlsx
Spoken:  0 results → BNC_help_spoken.xlsx


/tmp/ipykernel_11364/1892269304.py:141: DeprecationWarning: Testing an element's truth value will always return True in future versions.  Use specific 'len(elem)' or 'elem is not None' test instead.
  wstext_element = tree.find('.//wtext') or tree.find('.//stext')
